---
## Part 1. Sigmoid 함수 — 점수를 확률로 바꿔주는 변환기

### 왜 확률이 필요한가?

- 모델이 내부적으로 계산한 점수(z)는 **-∞ ~ +∞** 범위
- 하지만 우리가 원하는 건 **"이 사람이 생존할 확률이 몇 %인가?"**
- Sigmoid 함수가 이 점수를 **0 ~ 1 사이의 확률**로 바꿔줌

### Sigmoid 공식

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

| z 값 | sigmoid(z) | 해석 |
|------|-----------|------|
| -10 | ≈ 0.00 | 거의 확실히 음성 |
| 0 | 0.50 | 반반 (애매) |
| +10 | ≈ 1.00 | 거의 확실히 양성 |

> 마치 **시험 점수(z)**를 **합격 확률**로 바꿔주는 변환기!

In [1]:
import numpy as np 
import plotly.graph_objects as go 

z = np.linspace(-10, 10, 200)
##display(z)
##display(np.exp(-z)) ## 자연수를 거듭제곱(e^{x}) 값을 계산해 주는 지수 함수
sigmoid = 1 / (1 + np.exp(-z))
##display(sigmoid)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
    x=z, y=sigmoid, mode='lines',
    line=dict(color='purple', width=3), name='Sigmoid'
    )
)
fig.add_hline( y=0.5, line_dash='dash', line_color='grey', annotation_text='확률 0.5(기준선)')
fig.add_vline( x=0, line_dash='dash', line_color='grey')
fig.update_layout(title="sigmoid Function (점수 -> 확률 변환기)",
                  xaxis_title='z (모델 점수)',
                  yaxis_title='sigmoid(z) = 확률',
                  template ="plotly_dark",
                  yaxis=dict(range=[0.05, 1.05])
                  )
fig.show()

### Sigmoid 그래프 해석

- **z가 크면** → sigmoid ≈ 1 → "양성(생존)일 확률이 높다" (확신)
- **z가 작으면** → sigmoid ≈ 0 → "음성(사망)일 확률이 높다" (확신)
- **z = 0** → sigmoid = 0.5 → "반반, 가장 애매한 지점" (**결정 경계**)

> 핵심 1: **z = 0 ⇔ 확률 0.5** 가 기본 기준선
> 핵심 2: 경계 근처는 **민감(확률이 크게 흔들림)**, 양끝은 **둔감(0/1에 붙음)** → 헷갈림/확신이 생김

---
## Part 2. Titanic 데이터 전처리

### Titanic 데이터셋
- 실제 타이타닉호 생존자 데이터
- 목표: 승객의 나이, 성별, 선실 등급 등을 보고 **생존 여부(0=사망, 1=생존)** 예측
- 주요 컬럼:
  - `Pclass`: 선실 등급 (1=일등석, 2=이등석, 3=삼등석)
  - `Sex`: 성별
  - `Age`: 나이
  - `Fare`: 운임 요금
- **Target(정답)**: `Survived` (1=생존, 0=사망)

> 3회차에서 배운 전처리(인코딩, 결측치 처리)를 활용

In [3]:
import pandas as pd
from pathlib import Path

CSV = "titanic_train.csv" 
if not Path(CSV).exists():
    raise FileNotFoundError(
        f"{CSV} 를 이 노트북과 같은 폴더에 두고 다시 실행하세요!"
        "(Kaggle Titanic train.csv)"
    )

titanic_df = pd.read_csv(CSV)
display(titanic_df.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,1,"Palsson, Mrs. Jacques Heath",male,NaN,0,2,STON/O2. 279176,135.3813,NaN,S
1,2,1,3,"Braund, Master. Gosta Leonard",male,NaN,0,0,STON/O2. 979004,17.8233,E90,S
2,3,1,2,"Futrelle, Mr. James",male,35.9,0,0,STON/O2. 938583,28.3770,NaN,C
3,4,0,3,"Palsson, Mrs. Jacques Heath",female,NaN,0,0,A/5 960378,22.5759,F67,Q
4,5,0,3,"Heikkinen, Mrs. Oscar W",male,NaN,0,0,444676,22.4839,NaN,S


In [22]:
import pandas as pd 

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1).copy()
X_titanic_df['Cabin1'] = X_titanic_df['Cabin'].fillna("N").astype(str).str[:1] #객실번호의 첫부분의 구역정보만 추출 , null값은 N처리
X_titanic_df = X_titanic_df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)

#display(X_titanic_df)
print(f"데이터 크기: {X_titanic_df.shape}")
print(f"생존 비율: \n{y_titanic_df.value_counts(normalize=True)}")
display(X_titanic_df.head())

#1단계 최종 시험지(test)를 먼저 떼어내고 봉인함
X_dev, X_final_test, y_dev, y_final_test = train_test_split(
    X_titanic_df, y_titanic_df,
    test_size=0.2, stratify=y_titanic_df, random_state=42
)

#2단계 남은 데이터를 다시 교과서(train) / 모의고사(valid)로 나눔
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev,
    test_size=0.25, stratify=y_dev, random_state=42
)

print(f"train (교과서)   : {len(X_train)}명")
print(f"valid  (모의고사  : {len(X_valid)}명")
print(f"test    (수능)   : {len(X_final_test)}명  <- 맨 마지막에 딱 한 번만 씀")

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked", "Cabin1"]

numeric_transformer = Pipeline( steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline( steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop" 
)
lr = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=5000, random_state=42)),
])

lr.fit(X_train, y_train )
y_proba = lr.predict_proba(X_valid)[:,1]
display(pd.DataFrame({"예측 확률 (생존) 10개보기": y_proba[:10].round(4)}))

lr_model = lr.named_steps["model"]

print(f"solver         : {lr_model.solver}")
print(f"max_iter (설정) : {lr_model.max_iter}")
print(f"n_iter_  (실제) : {lr_model.n_iter_[0]}")


from sklearn.metrics import accuracy_score

y_pred = lr.predict(X_valid)
print(f"\n기본 Accuracy (threshold=0.5): {accuracy_score(y_valid, y_pred):.3f}")

데이터 크기: (100, 8)
생존 비율: 
Survived
0    0.66
1    0.34
Name: proportion, dtype: float64


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin1
0,1,male,NaN,0,2,135.3813,S,N
1,3,male,NaN,0,0,17.8233,S,E
2,2,male,35.9,0,0,28.3770,C,N
3,3,female,NaN,0,0,22.5759,Q,F
4,3,male,NaN,0,0,22.4839,S,N


train (교과서)   : 60명
valid  (모의고사  : 20명
test    (수능)   : 20명  <- 맨 마지막에 딱 한 번만 씀


,예측 확률 (생존) 10개보기
0,0.4383
1,0.6722
2,0.3141
3,0.3370
4,0.4298
5,0.4077
6,0.2693
7,0.3832
8,0.4161
9,0.3678


solver         : lbfgs
max_iter (설정) : 5000
n_iter_  (실제) : 16

기본 Accuracy (threshold=0.5): 0.700


학습과정 5단계
1. 직선을 아무렇게나 긋는다
2. 각 데이터의 확률을 계산한다.
3. 정답과 비교해서 벌점(Cross Entorypy)을 받는다
4. 벌점이 줄어드는 방향으로 직선을 조금 움직인다.
5. 이걸 손실이 더 줄지 않을 때까지 반복 --> 최적 직선에 도착!!
--> 확률 예측 오차(Cross Entropy)를 최소화하다 보면 결과적으로 잘 누는 선이 생긴다.

In [23]:
import numpy as np
from sklearn.metrics import roc_auc_score

def dummy_rule(X):
    """3-2회차 Part 1의 규칙: 남자면 사망(0), 아니면 생존(1)"""
    return np.where(X["Sex"].values == "male", 0, 1)

rows = []
for name, Xs, ys in [("valid", X_valid, y_valid)]:
    majority = np.zeros(len(ys), dtype=int)
    rows.append(["① 전부 사망 (다수 클래스)", round(accuracy_score(ys, majority), 4), "-"])
    rows.append(["② 3-2 Dummy 규칙 (남자=사망)", round(accuracy_score(ys, dummy_rule(Xs)), 4), "확률 없음"])
    rows.append(["③ 로지스틱 회귀 (피처 8개)",
                 round(accuracy_score(ys, (y_proba >= 0.5).astype(int)), 4),
                 round(roc_auc_score(ys, y_proba), 4)])

display(pd.DataFrame(rows, columns=["모델", "valid Accuracy", "AUC"]))

,모델,valid Accuracy,AUC
0,① 전부 사망 (다수 클래스),0.65,-
1,② 3-2 Dummy 규칙 (남자=사망),0.50,확률 없음
2,③ 로지스틱 회귀 (피처 8개),0.70,0.6154


In [27]:
y_proba 

array([0.43832503, 0.67224358, 0.31413391, 0.33704295, 0.42982609,
       0.40771506, 0.26931397, 0.38318797, 0.41613375, 0.3677684 ,
       0.31021379, 0.3854537 , 0.35058375, 0.35590079, 0.34082819,
       0.47304021, 0.20550609, 0.27712888, 0.35679552, 0.26422534])

In [28]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import pandas as pd

for thr in [0.3, 0.5, 0.7]:
    y_hat = (y_proba >= thr).astype(int)

    print(f"Threshold={thr}")
    print(f"  Precision: {precision_score(y_valid, y_hat, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_valid, y_hat, zero_division=0):.4f}")

    cm = confusion_matrix(y_valid, y_hat)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0 (사망)", "Actual 1 (생존)"],
        columns=["Pred 0 (사망)", "Pred 1 (생존)"]
    )
    display(cm_df)

    tn, fp, fn, tp = cm.ravel()
    print(f"  FP(오탐): {fp}   FN(미탐/누락): {fn}   TP: {tp}   TN: {tn}")
    print("-" * 40)

Threshold=0.3
  Precision: 0.3750
  Recall   : 0.8571


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),3,10
Actual 1 (생존),1,6


  FP(오탐): 10   FN(미탐/누락): 1   TP: 6   TN: 3
----------------------------------------
Threshold=0.5
  Precision: 1.0000
  Recall   : 0.1429


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),13,0
Actual 1 (생존),6,1


  FP(오탐): 0   FN(미탐/누락): 6   TP: 1   TN: 13
----------------------------------------
Threshold=0.7
  Precision: 0.0000
  Recall   : 0.0000


,Pred 0 (사망),Pred 1 (생존)
Actual 0 (사망),13,0
Actual 1 (생존),7,0


  FP(오탐): 0   FN(미탐/누락): 7   TP: 0   TN: 13
----------------------------------------


In [29]:
from sklearn.metrics import f1_score
import plotly.graph_objects as go

thr_list = np.round(np.arange(0.1, 1.0, 0.1), 2)
rows = []
for thr in thr_list:
    y_hat = (y_proba >= thr).astype(int)
    rows.append([
        thr,
        precision_score(y_valid, y_hat, zero_division=0),
        recall_score(y_valid, y_hat, zero_division=0),
        f1_score(y_valid, y_hat, zero_division=0)
    ])

thr_df = pd.DataFrame(rows, columns=["Threshold", "Precision", "Recall", "F1"])
display(thr_df)

,Threshold,Precision,Recall,F1
0,0.1,0.350,1.000000,0.518519
1,0.2,0.350,1.000000,0.518519
2,0.3,0.375,0.857143,0.521739
3,0.4,0.500,0.428571,0.461538
4,0.5,1.000,0.142857,0.250000
5,0.6,1.000,0.142857,0.250000
6,0.7,0.000,0.000000,0.000000
7,0.8,0.000,0.000000,0.000000
8,0.9,0.000,0.000000,0.000000
